# Distribution analysis

Every row of `dh Transaction Data` is one **store x week x SKU** observation. This notebook
characterises the distribution of the six key metrics so that later modelling choices
(which test, which error distribution, which transformation) are made against evidence
rather than assumption.

The three questions it answers:

1. **Where is the mass?** — location, spread and tail concentration per metric.
2. **What shape is it?** — normal? log-normal? something heavier?
3. **Does the shape change by group?** — promotion, week, store, SKU.

> Every numeric claim in the prose below is printed by the cell it follows, so it can be
> checked against the source data.

## 1. Setup & data

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (13, 4.5)
plt.rcParams["axes.titlesize"] = 12
plt.rcParams["axes.titleweight"] = "bold"

DATA = Path("../data/dunnhumby - Breakfast at the Frat.xlsx")
pd.set_option("display.float_format", lambda v: f"{v:,.2f}")
pd.set_option("display.max_columns", 50)

print(f"pandas {pd.__version__} | seaborn {sns.__version__} | scipy {stats.__name__}")

In [ ]:
xl = pd.ExcelFile(DATA, engine="openpyxl")

tx = pd.read_excel(xl, "dh Transaction Data", header=1)
stores = pd.read_excel(xl, "dh Store Lookup", header=1)
products = pd.read_excel(xl, "dh Products Lookup", header=1)

metrics = ["UNITS", "VISITS", "HHS", "SPEND", "PRICE", "BASE_PRICE"]
dfm = tx[metrics].copy()

print(f"tx: {tx.shape[0]:,} rows x {tx.shape[1]} cols  ({tx.WEEK_END_DATE.nunique()} weeks, "
      f"{tx.STORE_NUM.nunique()} stores, {tx.UPC.nunique()} SKUs)")
print("\nmissing values per metric:")
print(dfm.isna().sum().to_frame("n_missing").T)

In [ ]:
glossary = pd.read_excel(xl, "Glossary", header=3).drop(columns=["Unnamed: 3"])
glossary[glossary["VARIABLE NAME"].isin(metrics + ["WEEK_END_DATE", "STORE_NUM", "UPC",
                                                    "DISPLAY", "FEATURE", "TPR_ONLY"])]

## 2. Where the mass is

In [ ]:
pcts = [0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99, 0.999]
desc = dfm.describe(percentiles=pcts).T.rename(
    columns={"count": "n", "50%": "median", "75%": "p75", "25%": "p25",
             "5%": "p5", "10%": "p10", "90%": "p90", "95%": "p95", "99%": "p99",
             "99.9%": "p99.9"})

desc["n"] = desc["n"].astype(int)
desc["IQR"] = desc["p75"] - desc["p25"]
desc["MAD"] = dfm.apply(lambda s: (s - s.median()).abs().median())
desc["CV"] = desc["std"] / desc["mean"].replace(0, np.nan)
desc["skew"] = dfm.skew()
desc["kurtosis"] = dfm.kurt()
desc["pct_zero"] = (dfm == 0).mean() * 100
desc["var/mean"] = dfm.var() / dfm.mean()

order = ["n", "mean", "median", "std", "CV", "min", "p5", "p10", "p25", "p75",
         "p90", "p95", "p99", "p99.9", "max", "IQR", "MAD", "skew", "kurtosis",
         "pct_zero", "var/mean"]
desc = desc.reindex(columns=order)

with pd.option_context("display.float_format", lambda v: f"{v:,.3f}"):
    display(desc)

**Volume metrics** (`UNITS`, `VISITS`, `HHS`, `SPEND`) are strongly right-skewed: the mean sits
well above the median (`UNITS` 19.61 vs 10.00), `CV` is 1.28–1.53, and skew/kurtosis reach
6.84 / 114.31 for `UNITS`. `MAD` (7) is a third of the `IQR` (20), so the typical row is much
closer to the median than the standard deviation suggests — **report medians and IQR, not
mean ± sd**. These are also discrete counts and heavily overdispersed (`var/mean` = 35–46,
against 1.0 for a Poisson), which rules out a naive Poisson GLM.

**Price metrics** behave differently. `PRICE` / `BASE_PRICE` are bounded, near-symmetric
(skew 0.76 / 0.73, kurtosis −0.05 / −0.31) with `CV` ≈ 0.46. They are *not* heavy-tailed like the
volumes — only 699 distinct price points, and the single most common price ($2.99) covers 4.4%
of rows. Exactly **one** row has `PRICE == 0`, so there is no zero-inflation problem in practice.

Two side notes the table settles: the mean actual price ($3.38) sits below the mean base price
($3.60) because ~25.7% of rows are discounted, and `VISITS`/`HHS` are near-duplicates
(correlation 0.9986; `HHS` never exceeds `VISITS`, and equals it in 81.9% of rows) — one of the
two can be dropped from any model.

## 3. What shape is it?

Two views per metric, each covering all six metrics at once. Log views use `log1p` so zeros
stay defined.

In [ ]:
fig, axes = plt.subplots(2, len(metrics), figsize=(3.5 * len(metrics), 9))

for j, col in enumerate(metrics):
    raw = dfm[col].dropna()
    s = np.log1p(raw)

    # --- linear x-axis ---
    sns.histplot(raw, bins=60, kde=True, stat="density", color="steelblue",
                 ax=axes[0, j], line_kws={"lw": 1.5})
    axes[0, j].set_title(col)
    axes[0, j].set_xlabel(col)
    axes[0, j].set_ylabel("density")

    # --- log1p x-axis ---
    sns.histplot(s, bins=60, kde=True, stat="density", color="steelblue",
                 ax=axes[1, j], line_kws={"lw": 1.5})
    axes[1, j].axvline(0, color="gray", ls=":", lw=1)
    axes[1, j].set_xlabel(f"log1p({col})")
    axes[1, j].set_ylabel("density")

fig.suptitle("Raw vs log1p distribution shape", y=1.0, fontsize=13, fontweight="bold")
print(f"Spikes in raw data interval: {np.expm1(0)} - {np.expm1(2):.1f}")

fig.tight_layout()
plt.show()

The charts show both the linear and the `log1p` axis, for a cleaner read of each shape:

- `UNITS` - on the linear axis the peak sits between 0 and 500 units. The spikes are a consequence of discrete counts: at low values (0-5 units) `log1p` cannot smooth the curve, which only becomes smooth from about 6 units upward. The mean sits near ~20 units (raw approximation), consistent with the `describe()` table.
- `VISITS` / `HHS` - note that `log1p` is not the best approximation here: it does not settle the question of whether visits and active households are equal. That deserves a formal statistical test rather than an eyeball on the plot. The distribution, the spikes and the mean are nonetheless very similar to `UNITS`.
- `SPEND` - here the fit is close to perfect, consistent with the log-normal story for spend. Values concentrate roughly in the $6-53 band (raw plot approximation).
- `PRICE` / `BASE_PRICE` - I will only scratch the surface here, as the business detail is not the point of this notebook. We already know from the cell above that actual price usually sits below base price; the distribution tells us the business sells individual units below $10.

In [ ]:
# standardized probability plot: both transforms are put on a z-scale so the raw series
# (max ~1800 units) cannot flatten the log1p series into an invisible line
fig, axes = plt.subplots(2, 3, figsize=(15, 7.5))
for ax, col in zip(axes.ravel(), metrics):
    s = dfm[col].dropna()
    for vals, color, label in ((s, "steelblue", "raw"), (np.log1p(s), "seagreen", "log1p")):
        theo, samp = stats.probplot(vals, dist="norm")[0][:2]
        z = (samp - samp.mean()) / samp.std()
        keep = np.abs(theo) <= 3.5          # central 99.8% — beyond this one point sets the scale
        ax.plot(theo[keep], z[keep], color=color, lw=1.3, label=label)
    ax.plot([-4.5, 4.5], [-4.5, 4.5], color="crimson", lw=1, ls="--")
    ax.set(xlim=(-4.5, 4.5), ylim=(-4.5, 4.5),
           xlabel="theoretical quantile (z)", ylabel="sample quantile (z)")
    ax.set_title(col)
    ax.legend(loc="upper left", fontsize=8)
fig.suptitle("Standardized Q-Q vs the normal — closer to the red line = better fit",
             y=1.0, fontsize=13, fontweight="bold")
fig.tight_layout()
plt.show()

In [ ]:
rows = []
for col in metrics:
    s = dfm[col].dropna()
    r = {"metric": col}
    for tag, x in (("raw", s.to_numpy()), ("log1p", np.log1p(s.to_numpy()))):
        theo, samp = stats.probplot(x, dist="norm")[0][:2]
        r[f"R2_{tag}"] = np.corrcoef(theo, samp)[0, 1] ** 2
        r[f"skew_{tag}"] = stats.skew(x)
        r[f"kurt_{tag}"] = stats.kurtosis(x)
    rows.append(r)
fit = pd.DataFrame(rows).set_index("metric")

with pd.option_context("display.float_format", lambda v: f"{v:,.4f}"):
    display(fit)

`R2` is the squared correlation between theoretical and sample normal quantiles — a cheap,
distribution-free read on how well each transform fits a normal.

The picture is unambiguous. **Raw** volumes score `R2` 0.55–0.64 and arc far above the
reference line: they are nowhere near normal. **`log1p`** lifts them to 0.978–0.993 with skew
of −0.03 to +0.25. So a log response is close to a sufficient summary for all four volume
metrics — but note the residual kurtosis is ≈ −0.5 (slightly *flat*), so log-normal is an
approximation, not an exact family.

Prices are the exception and should stay untransformed: raw `PRICE` / `BASE_PRICE` already sit
at `R2` 0.94 / 0.93 with kurtosis ≈ 0, and the bounding at $11.46 means the tail is censored
rather than heavy.

## 4. Tails

How much of the total does the top of each distribution actually control?

In [ ]:
rows = []
for col in ["UNITS", "VISITS", "HHS", "SPEND"]:
    s = dfm[col].dropna()
    total = s.sum()
    r = {"metric": col}
    for p in (0.01, 0.05, 0.10):
        thr = s.quantile(1 - p)
        r[f"% of total in top {p:.0%} rows"] = s[s >= thr].sum() / total * 100
    r["max"] = s.max()
    r["p99.9"] = s.quantile(0.999)
    rows.append(r)
tail_tbl = pd.DataFrame(rows).set_index("metric").round(2)
display(tail_tbl)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for ax, col in zip(axes, ["UNITS", "SPEND"]):
    s = np.log1p(dfm[col].dropna())
    p95, p99, p999 = (np.log1p(dfm[col].quantile(q)) for q in (0.95, 0.99, 0.999))

    sns.histplot(s, bins=60, stat="density", color="steelblue", ax=ax)
    ax.set_xlim(0, s.max())
    ax.axvspan(p99, s.max(), color="crimson", alpha=0.07)
    top = ax.get_ylim()[1]
    for thr, lab, frac in ((p95, "p95", 0.98), (p99, "p99", 0.90), (p999, "p99.9", 0.82)):
        ax.axvline(thr, color="crimson", ls="--", lw=0.9)
        ax.text(thr, top * frac, lab, color="crimson", fontsize=8, rotation=90,
                ha="right", va="top")
    ax.set_xlabel(f"log1p({col})")
    ax.set_ylabel("density")
    ax.set_title(f"{col} — top 1% of rows shaded")
fig.suptitle("Where the tail begins", y=1.02, fontsize=13, fontweight="bold")
fig.tight_layout()
plt.show()

Tail concentration is real but **milder than "a handful of rows ruins everything"**: the top 1%
of rows holds 10.9% of `UNITS` (10× its share) and 8.7% of `SPEND`, and the top 10% holds only
43.6% / 38.7%. So the mean is pulled upward but no small minority dominates — trimming or
de-levering is optional here, not mandatory.

The tail is also short. The worst `UNITS` row (1,800) is only 5.7× the p99.9, versus the 100×+
ratios that justify winsorising in other retail panels. And the extremes are ordinary big
store-weeks rather than data errors: the ten largest rows span just 5 stores and 5 SKUs.

## 5. Does the shape change by group?

In [ ]:
tx_g = tx.copy()
tx_g["ANY_PROMO"] = ((tx_g.DISPLAY == 1) | (tx_g.FEATURE == 1) | (tx_g.TPR_ONLY == 1)).astype(int)
tx_g["PROMO"] = np.where(tx_g.ANY_PROMO == 1, "On promo", "No promo")

promo_flags = ["DISPLAY", "FEATURE", "TPR_ONLY", "ANY_PROMO"]
rate = (tx_g[promo_flags].mean() * 100).round(2).to_frame("% of rows")
display(rate)

In [ ]:
order = ["No promo", "On promo"]
palette = {"No promo": "#cccccc", "On promo": "#e8b33a"}
fig, axes = plt.subplots(2, 2, figsize=(13, 7))
for ax, col in zip(axes.ravel(), ["PRICE", "BASE_PRICE", "UNITS", "SPEND"]):
    sns.boxplot(data=tx_g, x="PROMO", y=col, order=order, hue="PROMO", legend=False,
                palette=palette, showfliers=False, ax=ax, linewidth=1.2)
    ax.set_yscale("log")
    ax.set_xlabel("")
    ax.set_ylabel(f"{col} (log scale)")
    med = tx_g.groupby("PROMO")[col].median()
    for i, lab in enumerate(order):
        ax.text(i, med[lab], f" {med[lab]:,.2f}", ha="center", va="center", fontsize=9,
                color="white", fontweight="bold")
    ax.set_title(f"{col} by promo  (median labelled)")
fig.suptitle("Volume and price by promotion status", y=1.0, fontsize=13, fontweight="bold")
fig.tight_layout()
plt.show()

In [ ]:
rows = []
for col in metrics:
    a = tx_g.loc[tx_g.ANY_PROMO == 1, col].dropna()
    b = tx_g.loc[tx_g.ANY_PROMO == 0, col].dropna()
    ks = stats.ks_2samp(a, b)
    rows.append({"metric": col,
                 "median no promo": b.median(), "median promo": a.median(),
                 "mean no promo": b.mean(), "mean promo": a.mean(),
                 "skew no promo": stats.skew(b), "skew promo": stats.skew(a),
                 "KS D": ks.statistic})
promo_tbl = pd.DataFrame(rows).set_index("metric").round(3)
display(promo_tbl)

`BASE_PRICE` merits close attention. The median is 3.17 with no promo and 3.19 under promo - the two are effectively identical. That means the comparison is not confounded by product composition: we are looking at goods that are equivalent in underlying nature, or at minimum a closely matched basket, with a stable base price.

The `SPEND` median rises from 29.34 to 41.51, and that is the most important result here. Even though revenue per unit declines, the final ticket - customer spend - rises by about 41 percent.

Volume growth of about 78 percent more than offsets the price decline of about 48 percent, so in raw arithmetic the business captures more revenue per transaction.

Notice that the yellow (promo) boxes for `UNITS` and `SPEND` are wider. On one side the campaign attracts very large buyers; on the other it may draw first-time buyers who purchase exactly one unit as a trial. Promotion makes customer behavior more heterogeneous - an interesting notion to inspect later.

In [ ]:
tx_t = tx.copy()
tx_t["WEEK_END_DATE"] = pd.to_datetime(tx_t.WEEK_END_DATE, errors="coerce")
print(f"{tx_t.WEEK_END_DATE.nunique()} weeks: {tx_t.WEEK_END_DATE.min():%Y-%m-%d} -> "
      f"{tx_t.WEEK_END_DATE.max():%Y-%m-%d}")

fig, axes = plt.subplots(2, 1, figsize=(13, 7), sharex=True)
for ax, col in zip(axes, ["UNITS", "SPEND"]):
    q = tx_t.groupby("WEEK_END_DATE")[col].quantile([0.25, 0.50, 0.75, 0.95]).unstack()
    ax.fill_between(q.index, q[0.25], q[0.75], color="steelblue", alpha=0.2,
                    label="IQR (p25-p75)")
    ax.plot(q.index, q[0.50], color="steelblue", lw=1.4, label="median")
    ax.plot(q.index, q[0.95], color="crimson", lw=1.0, ls="--", label="p95")
    ax.set_ylabel(col)
    ax.legend(loc="upper right", fontsize=8, ncols=3)
axes[0].set_title("UNITS per row, by week")
axes[1].set_title("SPEND per row, by week")
fig.suptitle("Weekly distribution drift (median, IQR, p95)", y=1.0, fontsize=13,
             fontweight="bold")
fig.autofmt_xdate()
fig.tight_layout()
plt.show()

This graph shows the sales curve for 2009 to 2012. The median and the interquartile band (the light blue zone) are very stable, practically unchanging. Neither seasonality nor a trend line is visible - although the aggregation level is high, so it is hard to say - and I will return to that topic in more detail in the main notebook. This is expected for a staple category, where demand is stable regardless of season. What we observe is a stable plateau in consumer behavior, indicating an established base of loyal customers with a stable purchase pattern.

Of interest is the red dashed line: sales well beyond the usual customer ticket. Here sales are much more volatile and deserve a detailed examination for trends and seasonality.

In [ ]:
# the store lookup ships 2 duplicated STORE_ID rows, so dedupe before mapping onto tx
store_lbl = (stores.drop_duplicates("STORE_ID")
             .set_index("STORE_ID")[["STORE_NAME", "SEG_VALUE_NAME"]])
sku_lbl = products.drop_duplicates("UPC").set_index("UPC")["DESCRIPTION"]

top_stores = tx_t.STORE_NUM.value_counts().head(8).index
top_upcs = tx_t.UPC.value_counts().head(8).index

sd = tx_t[tx_t.STORE_NUM.isin(top_stores)].join(store_lbl, on="STORE_NUM")
sd["LABEL"] = sd.STORE_NAME + " (" + sd.SEG_VALUE_NAME + ")"
kd = tx_t[tx_t.UPC.isin(top_upcs)].join(sku_lbl, on="UPC")
kd["LABEL"] = kd.DESCRIPTION.str.slice(0, 24)

fig, axes = plt.subplots(2, 2, figsize=(15, 8))
for col, ax in zip(["UNITS", "SPEND"], axes[0]):
    sns.boxplot(data=sd, x="LABEL", y=col, order=sd.LABEL.drop_duplicates(),
                color="steelblue", showfliers=False, ax=ax)
    ax.set_yscale("log"); ax.set_xlabel(""); ax.set_ylabel(f"{col} (log)")
    ax.set_title(f"{col} — top 8 stores by row count")
    ax.tick_params(axis="x", rotation=40, labelsize=7)
for col, ax in zip(["UNITS", "SPEND"], axes[1]):
    sns.boxplot(data=kd, x="LABEL", y=col, order=kd.LABEL.drop_duplicates(),
                color="seagreen", showfliers=False, ax=ax)
    ax.set_yscale("log"); ax.set_xlabel(""); ax.set_ylabel(f"{col} (log)")
    ax.set_title(f"{col} — top 8 SKUs by row count")
    ax.tick_params(axis="x", rotation=40, labelsize=7)
fig.suptitle("Cross-sectional heterogeneity (fliers omitted)", y=1.0, fontsize=13,
             fontweight="bold")
fig.tight_layout()
plt.show()

Pronounced cross-sectional heterogeneity is observed at both the store level and the SKU level.

Medians and volatility differ across them. However, contrary to expectations, belonging to the 'Upscale' segment does not guarantee a higher median ticket - Hyde Park (Upscale) sits below Cincinnati (Mainstream). This calls for testing the hypothesis that other factors influence the ticket, such as the size of the retail space. Note that the other obvious candidate, `PARKING_SPACE_QTY`, is missing on 52 of 79 store rows, so it is unusable as a feature as it stands.

The distributions show a different nature of demand: goods with a high median `UNITS` (Cheerios) have a `SPEND` comparable to goods where `UNITS` is lower. This indicates differences in price positioning within the basket.

## 6. Conclusion

**Where the mass is.** The four volume metrics are discrete, right-skewed and heavily overdispersed — median `UNITS` 10.00 against a mean of 19.61, `MAD` 7 versus `IQR` 20, `var/mean` 35–46. Medians and IQRs describe them honestly where mean ± sd does not, and no Poisson assumption survives the variance. `PRICE` / `BASE_PRICE` are a different animal: bounded at $11.46, near-symmetric (skew 0.76 / 0.73, kurtosis −0.05 / −0.31), 699 distinct price points, and exactly one row at `PRICE == 0`.

**What shape it is.** `log1p` lifts the volumes from `R2` 0.55–0.64 to 0.978–0.993, with skew inside −0.03 to +0.25. The residual kurtosis of ≈ −0.5 says log-normal is close to sufficient, not exact. Prices need no transform — raw `R2` is 0.94 / 0.93 with kurtosis ≈ 0.

**Tails.** Real but not dominant: the top 1% of rows holds 10.9% of `UNITS` and 8.7% of `SPEND`; the worst row (1,800 units) is 5.7× p99.9, and the ten largest rows come from just 5 stores and 5 SKUs — ordinary big store-weeks, not data errors. Winsorising or trimming is optional in a first pass.

**What changes by group.** Promotion moves the location, not the basket: `BASE_PRICE` medians are effectively identical (3.17 vs 3.19) while `SPEND` goes 29.34 → 41.51, so volume growth more than offsets the price cut and the ticket rises. It also moves the dispersion — the promo boxes are wider. Across time, the weekly median and IQR are flat 2009–2012 with no visible seasonality or trend at this aggregation, while the p95 line moves noticeably. Across stores and SKUs heterogeneity is pronounced, and belonging to the 'Upscale' segment does not by itself imply a larger ticket (Hyde Park sits below Cincinnati in median).

This notebook covers technical details, so the business insights and analysis will come in main notebook

**Open threads.**

- Test `VISITS` / `HHS` formally instead of by eye — correlation 0.9986, identical in 81.9% of rows. Then why not keep only one of the two.
- Split promo buyers into large-ticket versus one-unit trial, to test whether "more heterogeneous" is really a buyer-mix effect.
- The weekly p95 line is where seasonality and trend should be hunted in the main notebook; the median is too flat to show them.
- Store ticket drivers: segment alone does not explain the ticket, so test retail-space size — and note `PARKING_SPACE_QTY` is missing on 52 of 79 store rows, so it cannot serve as a feature as it stands.